# Reinforcement Learning: Policy Gradients for a Unicycle Robot

This notebook trains a stochastic steering policy for a unicycle robot with the policy-gradient method from Lecture 8.

## Unicycle model

The state is $s = (x, y, \phi)$: the position $(x, y)$ and the heading angle $\phi$. The robot moves at constant forward speed $v=1$, and the action is its angular velocity $u$.

Using an Euler discretization with step $\delta$, and a small random steering disturbance $w_k \sim \mathcal{N}(0, \sigma^2)$:

$$
\begin{aligned}
x_{k+1} &= x_k + \delta \, v \cos\phi_k \\
y_{k+1} &= y_k + \delta \, v \sin\phi_k \\
\phi_{k+1} &= \phi_k + \delta \, u_k + w_k
\end{aligned}
$$

The robot starts at $(0, 0)$ with heading $\phi = 0$. The objective is to reach the goal $(x_g, y_g)$. Once the position of the robot first enters the circle of radius $\varepsilon=0.5$ m around the goal, we stop the robot and terminate the episode, awarding reward 1. Otherwise, the episode ends after at most $H+1$ actions.

The steering policy selects its angular velocity stochastically using a three-action softmax, as in Lecture 8.

In [155]:
import numpy as np
import matplotlib.pyplot as plt

In [156]:
delta = 0.1        # discretization step
steps = 100        # maximum number of simulation steps
x_goal, y_goal = 4.0, 3.0
NOISE_STD = 0.015  # steering noise standard deviation (radians)

## 1. Softmax policy using two features

The possible actions (angular velocities) are $\mathcal A=\{-u_{\max},0,+u_{\max}\}$. Define the signed **heading error**

$$
e(s)=\operatorname{wrap}_{[-\pi,\pi)}\!\left(\operatorname{atan2}(y_g-y,x_g-x)-\phi\right).
$$

It is zero when the robot points directly toward the goal. We use only a constant (bias) and the normalized heading error:

$$
f(s)=\begin{bmatrix}1\\e(s)/\pi\end{bmatrix},\qquad
\Theta\in\mathbb R^{3\times 2},\qquad
\pi_\Theta(a_j\mid s)=\frac{\exp(\theta_j^\top f(s))}{\sum_{\ell=1}^{3}\exp(\theta_\ell^\top f(s))}.
$$
At each time step the policy takes as input the state of the unicycle, namely its 2D position and heading angle, and gives the probability of choosing any of the three control inputs. This probability is parameterized by $\Theta \in \mathbb{R}^{3\times 2}$, which we aim to determine using policy gradient approach.
Note that unlike the previous closed-loop example, the policy does not output actions directly. It outputs a probability distribution over the three actions: steering left, straight, or right; we then sample one action: the angular velocity. 

Distance to the goal is not a policy feature: we use it only to detect success.

In [ ]:
U_MAX = 2.5             # angular velocity choices (rad/s)
ACTIONS = np.array([-U_MAX, 0.0, U_MAX])
GAMMA = 0.98
H = steps - 1           # Lecture 8: k=0,...,H, so H+1 = 100 actions
GOAL_RADIUS = 0.5      # terminal region; robot stops as soon as it enters

def state_features(s):
    x, y, phi = s
    desired = np.arctan2(y_goal-y, x_goal-x)
    error = (desired-phi+np.pi) % (2*np.pi) - np.pi
    return np.array([1.0, error/np.pi])

def softmax_policy(s, theta):
    logits = theta @ state_features(s)
    weights = np.exp(logits - np.max(logits))
    return weights / weights.sum()

# Rows: left (-U_MAX), straight (0), right (+U_MAX).
# Columns: [constant 1, signed heading error / pi].
theta = np.array([[0.0, -2.0],
                  [0.7,  0.0],
                  [0.0,  2.0]], dtype=float)
theta = np.zeros((ACTIONS.shape[0],state_features([0,0,0]).shape[0]))


print('Steering actions:', ACTIONS)
print('Initial-action probabilities:', np.round(softmax_policy([0., 0., 0.], theta), 3))

## 2. Sample trajectories and calculate discounted returns

At each step, draw $a_k\sim\pi_\Theta(\cdot\mid s_k)$ and simulate one time step with $v=1$. If the **new position** lies within the goal region, give reward 1 and stop; otherwise reward is zero:

$$
r(s_k,a_k,s_{k+1})=
\begin{cases}
1,&\|(x_{k+1},y_{k+1})-(x_g,y_g)\|\le\varepsilon,\\
0,&\text{otherwise},
\end{cases}\qquad \varepsilon=0.5\;\mathrm m.
$$

For a full horizon, Lecture 8 uses $\tau_H=(s_0,a_0,s_1,a_1,\ldots,s_H,a_H,s_{H+1})$ and

$$R(\tau_H)=\sum_{k=0}^{H}\gamma^k r(s_k,a_k,s_{k+1}).$$

In our code the sums simply terminate early if the robot reaches the goal. **Earlier success yields greater discounted return** when $0<\gamma<1$. If the robot never reaches the goal, its return is zero.

In [ ]:
def rollout(theta, rng, horizon=H+1):
    """Sample at most H+1 actions; stop robot immediately on first goal arrival."""
    s = np.array([0.0, 0.0, 0.0])
    states = [s.copy()]
    action_indices, rewards, score_grads = [], [], []
    reached_goal = False

    for k in range(horizon):
        features = state_features(s)
        probabilities = softmax_policy(s, theta)
        j = rng.choice(3, p=probabilities)
        u = ACTIONS[j]

        # Gradient of log pi_theta(a_j | s) for the three-action softmax.
        unit_vector_j = np.zeros(3)
        unit_vector_j[j] = 1.0
        score_grads.append(np.outer(unit_vector_j - probabilities, features))

        x, y, phi = s
        v = 1.0  # fixed forward velocity for this reinforcement-learning example
        steering_noise = rng.normal(0, NOISE_STD)
        s = np.array([x + delta*v*np.cos(phi),
                      y + delta*v*np.sin(phi),
                      phi + delta*u + steering_noise])

        d_next = np.hypot(x_goal - s[0], y_goal - s[1])
        reached_goal = (d_next <= GOAL_RADIUS)
        rewards.append(float(reached_goal))
        action_indices.append(j)
        states.append(s.copy())
        if reached_goal:
            break  # first arrival gives reward 1; terminate episode

    discounts = GAMMA ** np.arange(len(rewards))
    return {'states': np.array(states),
            'actions': np.array(action_indices),
            'rewards': np.array(rewards),
            'return': float(np.dot(discounts, rewards)),
            'success': reached_goal,
            'score_sum': np.sum(score_grads, axis=0),
            'min_distance': float(np.min(np.hypot(x_goal-np.array(states)[:,0], y_goal-np.array(states)[:,1])))}

N_TRAJECTORIES = 100
rng_rl = np.random.default_rng(2026)
trajectories = [rollout(theta, rng_rl) for _ in range(N_TRAJECTORIES)]
returns = np.array([tr['return'] for tr in trajectories])
successes = np.array([tr['success'] for tr in trajectories])
print(f'Goal radius: {GOAL_RADIUS} m; forward speed: 1 m/s')
print(f'Reached goal: {successes.sum()}/{N_TRAJECTORIES} trajectories ({successes.mean():.1%})')
print(f'Estimated expected discounted return: {returns.mean():.4f}')
print(f'Sample standard deviation of trajectory returns: {returns.std(ddof=1):.4f}')
print('First ten trajectory returns:', np.round(returns[:10], 3))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for tr in trajectories[:20]:
    ss = tr['states']
    axs[0].plot(ss[:, 0], ss[:, 1], alpha=0.45, linewidth=1)
axs[0].scatter([0], [0], c='black', marker='s', label='Start')
axs[0].scatter([x_goal], [y_goal], c='red', marker='*', s=180, label='Goal')
axs[0].add_patch(plt.Circle((x_goal, y_goal), GOAL_RADIUS, facecolor='red', alpha=0.12, edgecolor='red'))
axs[0].set(xlabel='x (m)', ylabel='y (m)', title='20 sampled trajectories under the same policy')
axs[0].axis('equal')
axs[0].grid(True, alpha=.3)
axs[0].legend()
axs[1].hist(returns, bins=14, edgecolor='white')
axs[1].axvline(returns.mean(), color='black', linestyle='--', label='Sample mean')
axs[1].set(xlabel='Discounted trajectory return', ylabel='Count',
           title=f'Returns from {N_TRAJECTORIES} trajectories')
axs[1].legend()
plt.tight_layout()
plt.show()

## 3. Estimate the policy gradient from sampled trajectories

For the three-action softmax, the gradient of the log-probability of choosing $a_j$ is

$$
\nabla_\Theta\log\pi_\Theta(a_j\mid s)
=\bigl(\hat{e}_j-\pi_\Theta(\cdot\mid s)\bigr)f(s)^\top.
$$

As on slides 7 and 14 of Lecture 8, the Monte Carlo estimate from $N$ trajectories sampled with the current parameters $\Theta_t$ is

$$
\boxed{\widehat{\nabla_\Theta J}(\Theta_t)=
\frac1N\sum_{i=1}^N
\left(\sum_{k=0}^{H}\gamma^k r(s_k^i,a_k^i,s_{k+1}^i)\right)
\left(\sum_{k=0}^{H}\nabla_\Theta\log\pi_{\Theta_t}(a_k^i\mid s_k^i)\right).}
$$

The sums stop early for a successful episode. No differentiation through the robot dynamics is needed.

In [ ]:
# Monte Carlo / REINFORCE policy gradient, as in Lecture 8, slide 14.
score_sums = np.stack([tr['score_sum'] for tr in trajectories])  # (N, 3, 2)
gradient_samples = returns[:, None, None] * score_sums
policy_gradient = gradient_samples.mean(axis=0)
print('Estimated policy gradient (rows=actions; columns=features):')
print(np.round(policy_gradient, 4))
print('Gradient matrix shape:', policy_gradient.shape)

# Sanity check: softmax probabilities are unchanged if the same vector
# is added to all three logit rows, so each column of the gradient sums to 0.
print('Column sums (should be near zero):', np.round(policy_gradient.sum(axis=0), 10))

## 4. One gradient-ascent update: what changes?

We use the gradient-ascent rule from Lecture 8:

$$\Theta_{t+1}=\Theta_t+\alpha\,\widehat{\nabla_\Theta J}(\Theta_t),$$

where $t$ counts parameter updates (not time steps, which are indexed by $k$).

A single small step changes the parameters and action probabilities, but **usually has too little effect to show clearly in a few stochastic trajectories**. First, inspect the numerical parameter/probability changes and evaluate many trajectories. Then, in the next section, compare policies after **0, 1, 5 and 15** updates to reveal the learning effect more clearly.

In [ ]:
# One update t=0 -> t=1 of the policy. Do not infer improvement from one sample path.
ALPHA = 0.1
theta_updated = theta + ALPHA * policy_gradient
sample_state = np.array([0., 0., 0.])
print('Action probabilities at initial state:')
print('Before update:', np.round(softmax_policy(sample_state, theta), 5))
print('After update: ', np.round(softmax_policy(sample_state, theta_updated), 5))

N_EVAL = 150
old_eval = [rollout(theta, np.random.default_rng(7000 + i)) for i in range(N_EVAL)]
new_eval = [rollout(theta_updated, np.random.default_rng(7000 + i)) for i in range(N_EVAL)]
for name, res in [('Before one update', old_eval), ('After one update', new_eval)]:
    print(f'{name}: success = {np.mean([z["success"] for z in res]):.1%}, '
          f'mean discounted return = {np.mean([z["return"] for z in res]):.3f}')
print('A one-step update is mathematically meaningful even if its practical effect is hard to see.')

## 5. Learning from a uniform policy: compare 0, 1, 5, and 15 updates

Start with $\Theta_0=0$, meaning the three steering actions are initially equally likely. At each iteration $t$, generate a **fresh batch** of trajectories with $\Theta_t$, estimate the Lecture 8 policy gradient, and perform one update to obtain $\Theta_{t+1}$.

Evaluate the policies $\Theta_t$ for $t=$ **0, 1, 5, and 15** updates on independent test trajectories, keeping all simulation settings unchanged. We plot success rate, average closest approach to the goal, and action probabilities. Finally, compare sample trajectories **side by side** (same set of random seeds) before training and after 15 updates; green means the robot entered the goal circle and stopped.

Only $u_{\max}$ was increased to $2.5$ rad/s relative to the earlier experiment. **Forward speed stays at $v=1$, the goal radius stays at 0.5 m, and the reward stays binary.**

In [ ]:
# Repeated policy-gradient updates, with H matching the notation from Lecture 8.
TRAIN_BATCH = 160
TRAIN_UPDATES = 15
TRAIN_LR = 1

train_theta = np.zeros((ACTIONS.shape[0], state_features([0,0,0]).shape[0]))
theta_history = [train_theta.copy()]
training_success = []
for t in range(TRAIN_UPDATES):
    rg = np.random.default_rng(1000 + t)
    batch = [rollout(train_theta, rg) for _ in range(TRAIN_BATCH)]
    grad = np.mean([tr['return'] * tr['score_sum'] for tr in batch], axis=0)
    training_success.append(np.mean([tr['success'] for tr in batch]))
    train_theta += TRAIN_LR * grad
    theta_history.append(train_theta.copy())

# Evaluate on independent episodes with identical random seeds for every policy.
CHECKPOINTS = [0, 1, 5, 15]
N_TEST = 300
checks = {}
for t in CHECKPOINTS:
    trials = [rollout(theta_history[t], np.random.default_rng(999+i)) for i in range(N_TEST)]
    checks[t] = dict(success=np.mean([tr['success'] for tr in trials]),
                     ret=np.mean([tr['return'] for tr in trials]),
                     min_dist=np.mean([tr['min_distance'] for tr in trials]))
    print(f'{t:2d} updates: success = {checks[t]["success"]:.1%}, '
          f'mean discounted return = {checks[t]["ret"]:.3f}, '
          f'mean minimum distance to goal = {checks[t]["min_dist"]:.2f} m')

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(np.arange(1, TRAIN_UPDATES+1), training_success, 'o-', label='Training batch success')
axs[0].plot(CHECKPOINTS, [checks[t]['success'] for t in CHECKPOINTS], 's-',
            markersize=8, linewidth=2, label='Independent evaluation')
axs[0].set(xlabel='Number of gradient steps', ylabel='Fraction reaching the goal',
           title='Learning with sparse reward')
axs[0].set_ylim(0,1); axs[0].grid(alpha=0.3); axs[0].legend()

# Plot only two curves of action probability at a representative positive heading error.
s_example = [0., 0., -0.6]  # goal direction is above the robot's current heading
probs = np.array([softmax_policy(s_example, theta_history[t]) for t in CHECKPOINTS])
for j, label in enumerate(['-u_max', '0', '+u_max']):
    axs[1].plot(CHECKPOINTS, probs[:, j], 'o-', label=label)
axs[1].set(xlabel='Number of gradient steps', ylabel='Probability',
           title='Action probabilities at a fixed state')
axs[1].set_ylim(0,1); axs[1].grid(alpha=0.3); axs[1].legend()
plt.tight_layout(); plt.show()

# Side-by-side sample trajectories, same set of seeds, avoiding a confusing overlay.
fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, t in zip(axs, [0, 15]):
    policy_theta = theta_history[t]
    successes = 0
    for i in range(20):
        tr = rollout(policy_theta, np.random.default_rng(30000+i))
        path = tr['states']; successes += tr['success']
        ax.plot(path[:,0], path[:,1], alpha=0.5, lw=1,
                color=('tab:green' if tr['success'] else 'tab:gray'))
        if tr['success']:
            ax.plot(path[-1,0], path[-1,1], marker='o', color='tab:green', ms=3)
    ax.add_patch(plt.Circle((x_goal,y_goal), GOAL_RADIUS, fill=False,
                             edgecolor='red', linewidth=2))
    ax.scatter([0], [0], c='black', marker='s', zorder=4)
    ax.scatter([x_goal], [y_goal], c='red', marker='*', s=120, zorder=4)
    ax.set(xlabel='x (m)', ylabel='y (m)',
           title=f'{t} updates: {successes}/20 displayed reach the goal')
    ax.grid(alpha=0.25); ax.set_aspect('equal', adjustable='box')
axs[0].set_xlim(-5, 12); axs[0].set_ylim(-7, 10)
plt.suptitle('Same simulation conditions; green = success, gray = no success')
plt.tight_layout(); plt.show()

**Interpretation.** Larger $u_{\max}$ permits tighter turns at the **same forward speed**. The robot keeps moving at $v=1$ unless it reaches the goal circle, at which point the simulation ends and the robot is considered stopped. A gray trajectory may pass *near* the goal without ever entering the 0.5 m circle.

The sparse reward makes learning difficult because unsuccessful trajectories provide no policy-gradient signal. One update illustrates the algebra; comparing policies after several updates is much more informative for seeing an improvement. **An individual stochastic path is not proof that a policy improved**—use the independent success-rate evaluation.